In [ ]:
"""
One person tells a neighbor (local improvement)
That neighbor tells another
Eventually the whole network knows (global convergence)

Bellman-Ford works because repeated local improvements propagate information through the graph until the full global picture emerges.
"""

In [1]:
import math

def initialize_single_source(vertices, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0
    return dist, pred

def relax(u, v, weight, dist, pred):
    if dist[u] + weight < dist[v]:
        dist[v] = dist[u] + weight
        pred[v] = u

def bellman_ford(vertices, edges, source):
    dist, pred = initialize_single_source(vertices, source)

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            relax(u, v, w, dist, pred)

    # negative-weight cycles
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            return False, None, None

    return True, dist, pred

In [2]:
vertices = ['A', 'B', 'C', 'D']
edges = [
    ('A', 'B', 1),
    ('B', 'C', 3),
    ('A', 'C', 10),
    ('C', 'D', -2),
    ('D', 'B', -1)
]

ok, dist, pred = bellman_ford(vertices, edges, 'A')

if ok:
    print("Distances:", dist)
    print("Predecessors:", pred)
else:
    print("Graph contains a negative-weight cycle")

Distances: {'A': 0, 'B': 1, 'C': 4, 'D': 2}
Predecessors: {'A': None, 'B': 'A', 'C': 'B', 'D': 'C'}


In [6]:
"""
Modify Bellman-Ford to terminate early if a full pass over all edges performs no relaxations.
Since shortest paths have at most m edges, the algorithm will stop after at most m+1 passes.
"""
import math

def bellman_ford(vertices, edges, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        updated = False

        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u
                updated = True

        if not updated:
            break   # EARLY TERMINATION
            
    return dist, pred

In [7]:
"""
After the standard Bellman-Ford relaxation, perform an additional pass
to identify vertices whose distances can still be improved, mark them as part
of or affected by a negative cycle, and propagate −∞ from these vertices to all
reachable vertices.
"""
def bellman_ford(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}

    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    affected = set()

    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            affected.add(v)

    from collections import deque

    queue = deque(affected)

    while queue:
        u = queue.popleft()

        if dist[u] != float('-inf'):
            dist[u] = float('-inf')

        for (a, b, w) in edges:
            if a == u and dist[b] != float('-inf'):
                dist[b] = float('-inf')
                queue.append(b)

    return dist, pred

In [8]:
"""
Bellman-Ford normally computes:
shortest paths FROM one source s

But here we want:
shortest paths TO v from ANY source

Instead of trying every possible source, we simulate "all sources at once"
by adding a fake node that connects to everything for free.

Add a new source vertex s* and connect it to every vertex v ∈ V with an edge of weight 0.
Then run Bellman-Ford with s* as the source. The resulting distances d[v] equal δ*(v) for all v.
The running time is O(VE).
"""
def compute_delta_star(vertices, edges):
    s_star = "super_source"

    new_edges = edges.copy()
    for v in vertices:
        new_edges.append((s_star, v, 0))

    dist = {v: float('inf') for v in vertices}
    dist[s_star] = 0

    for _ in range(len(vertices)):
        for (u, v, w) in new_edges:
            if dist.get(u, float('inf')) + w < dist.get(v, float('inf')):
                dist[v] = dist[u] + w

    return {v: dist[v] for v in vertices}

In [10]:
"""
The algorithm does not return to the source
It returns a closed loop of vertices forming a negative cycle

The loop is found by:
detecting a bad vertex
walking into the cycle
looping until you return to the start

You follow predecessor pointers until they repeat because repetition means you've found a cycle.
"""
def find_negative_cycle(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    # negative cycle
    x = None
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            x = v
            break

    if x is None:
        return None  # No negative cycle

    for _ in range(len(vertices)):
        x = pred[x]

    cycle = []
    start = x

    while True:
        cycle.append(x)
        x = pred[x]
        if x == start:
            cycle.append(x)
            break

    cycle.reverse()
    return cycle

In [11]:
def topological_sort(G):
    visited = set()
    stack = []

    def dfs(u):
        visited.add(u)
        for v in G[u]:
            if v not in visited:
                dfs(v)
        stack.append(u)

    for u in G:
        if u not in visited:
            dfs(u)

    return stack[::-1]  # reverse to get correct order

In [12]:
def dag_shortest_paths(G, w, s):
    topo_order = topological_sort(G)
    
    d = {v: float('inf') for v in G}
    pi = {v: None for v in G}
    d[s] = 0
    
    for u in topo_order:
        for v in G[u]:
            if d[v] > d[u] + w[(u, v)]:
                d[v] = d[u] + w[(u, v)]
                pi[v] = u
    
    return d, pi

In [13]:
G = {
    'r': ['s', 't'],
    's': ['t', 'x'],
    't': ['x', 'y', 'z'],
    'x': ['y', 'z'],
    'y': ['z'],
    'z': []
}

w = {
    ('r','s'): 5, ('r','t'): 3,
    ('s','t'): 2, ('s','x'): 6,
    ('t','x'): 7, ('t','y'): 4, ('t','z'): 2,
    ('x','y'): -1, ('x','z'): 1,
    ('y','z'): -2
}

d, pi = dag_shortest_paths(G, w, 'r')

print("Distances:", d)
print("Predecessors:", pi)

Distances: {'r': 0, 's': 5, 't': 3, 'x': 10, 'y': 7, 'z': 5}
Predecessors: {'r': None, 's': 'r', 't': 'r', 'x': 't', 'y': 't', 'z': 't'}


In [15]:
"""
Tasks = vertices
Dependencies = edges
Output = minimum time to finish the project

We flipped shortest-path into longest-path to find the bottleneck chain of tasks,
which determines total completion time.
"""
def dag_longest_paths(G, weight, s):
    topo_order = topological_sort(G)
    
    d = {v: float('-inf') for v in G}
    pi = {v: None for v in G}
    d[s] = weight[s]
    
    for u in topo_order:
        for v in G[u]:
            if d[v] < d[u] + weight[v]:   # MAX instead of MIN
                d[v] = d[u] + weight[v]
                pi[v] = u
    
    return d, pi

In [16]:
G = {
    'A': ['B', 'C'],
    'B': ['D'],
    'C': ['D'],
    'D': ['E'],
    'E': []
}

weight = {
    'A': 3,
    'B': 2,
    'C': 4,
    'D': 2,
    'E': 1
}

source = 'A'

print(dag_longest_paths(G, weight, source))

({'A': 3, 'B': 5, 'C': 7, 'D': 9, 'E': 10}, {'A': None, 'B': 'A', 'C': 'A', 'D': 'C', 'E': 'D'})


In [17]:
"""
Use reverse topological DP where each node counts all paths starting from it.

This is not summing weights it's summing recursively computed path counts,
which encode exponentially many paths.
"""
def count_all_paths(G):
    topo_order = topological_sort(G)
    
    dp = {u: 0 for u in G}
    
    for u in reversed(topo_order):
        dp[u] = 1
        for v in G[u]:
            dp[u] += dp[v]
    
    total_paths = sum(dp.values())
    
    return total_paths

In [18]:
"""
Lazy update (what Python actually does)
heapq does NOT support decrease-key.

So instead:
When you find a better distance, you push a new entry
You don't remove the old one
Later, when popping, you ignore outdated entries

Lazy update = We allow duplicates and clean them up later
Decrease-key = We surgically update the existing entry
"""
import heapq

def dijkstra(G, w, s):
    dist, pred = initialize_single_source(G['V'], s)

    Q = []
    heapq.heappush(Q, (0, s))

    while Q:
        current_dist, u = heapq.heappop(Q)

        if current_dist > dist[u]:
            continue

        for v in G['adj'][u]:
            if relax(u, v, w, dist, pred):
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [19]:
G = {
    'V': ['A', 'B', 'C', 'D'],
    'adj': {
        'A': ['B', 'C'],
        'B': ['C', 'D'],
        'C': ['D'],
        'D': []
    }
}

w = {
    ('A', 'B'): 1,
    ('A', 'C'): 4,
    ('B', 'C'): 2,
    ('B', 'D'): 5,
    ('C', 'D'): 1
}



In [20]:
"""
Dijkstra can lock in wrong values early because it assumes no future improvement.

Dijkstra (greedy)
Always picks closest unvisited node

Assumes:
Once I pick a node, its distance is FINAL
Works only if all weights ≥ 0

Bellman-Ford (systematic)
Repeatedly relaxes ALL edges
Does this |V|-1 times
Works even with negative weights

Dijkstra
I trust my current best choice will never improve later

Bellman-Ford
I don't trust anything, I'll keep checking everything repeatedly
"""

"\nDijkstra can lock in wrong values early because it assumes no future improvement.\n\nDijkstra (greedy)\nAlways picks closest unvisited node\n\nAssumes:\nOnce I pick a node, its distance is FINAL\nWorks only if all weights ≥ 0\n\nBellman-Ford (systematic)\nRepeatedly relaxes ALL edges\nDoes this |V|-1 times\nWorks even with negative weights\n\nDijkstra\nI trust my current best choice will never improve later\n\nBellman-Ford\nI don't trust anything, I'll keep checking everything repeatedly\n"

In [21]:
"""
| BFS                 | Modified Dijkstra         |
| ------------------- | ------------------------- |
| Queue (FIFO)        | Priority queue (min-heap) |
| Add when discovered | Add when relaxed          |
| No duplicates       | May have duplicates       |

More efficient for sparse reachable regions
Avoids inserting unreachable vertices
Cleaner mental model (closer to BFS)

You can safely restrict Q to only discovered vertices,
as long as you insert vertices when they are first relaxed and maintain the min-priority property.
"""
import heapq
import math

def dijkstra(G, w, s):
    dist = {v: math.inf for v in G['V']}
    pred = {v: None for v in G['V']}

    dist[s] = 0

    Q = []
    heapq.heappush(Q, (0, s))

    visited = set() # S (S is set that we refer to in CLRS book)

    while Q:
        d_u, u = heapq.heappop(Q)

        if u in visited:
            continue

        visited.add(u)

        for v in G['adj'][u]:
            if dist[v] > dist[u] + w[(u, v)]:
                dist[v] = dist[u] + w[(u, v)]
                pred[v] = u
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [22]:
"""
Dijkstra explores vertices in order of current best distance
That order can include vertices not on the shortest path
So relaxations along the shortest path can be interleaved or delayed
"""

'\nDijkstra explores vertices in order of current best distance\nThat order can include vertices not on the shortest path\nSo relaxations along the shortest path can be interleaved or delayed\n'

In [23]:
"""
Maximizing a product = minimizing the negative log of that product
we can transform weights by minimizing the negative log of that product

Since 0 <= r(u,v) <= 1, we have:
log(r) <= 0
so −log(r) >= 0

All edge weights are nonnegative
That means we can safely use Dijkstra

If you want the actual probability:
reliability = math.exp(-dist[t])


Why maximizing becomes minimizing

For a path P:
r(P) = ∏r(u,v)

Take logs:
logr(P) = ∑logr(u,v)

Now multiply by -1:
−logr(P) = ∑−logr(u,v)

So:
maximizing r(P)
is equivalent to minimizing −logr(P)

argmax product : argmin sum of transformed weights

−log doesn't distort the problem it converts multiplicative probabilities
into additive costs so Dijkstra can solve it, and exponentiating restores the original probability.
"""
import math
import heapq

def most_reliable_path(G, r, s, t):
    w = {(u,v): -math.log(r[(u,v)]) for (u,v) in r}  # transform weights

    # Step 2: Dijkstra
    dist = {v: float('inf') for v in G['V']}
    pred = {v: None for v in G['V']}
    dist[s] = 0

    Q = [(0, s)]

    while Q:
        d_u, u = heapq.heappop(Q)

        if u == t:
            break

        if d_u > dist[u]:
            continue

        for v in G['adj'][u]:
            if dist[v] > dist[u] + w[(u,v)]:
                dist[v] = dist[u] + w[(u,v)]
                pred[v] = u
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [24]:
"""
using buckets instead of a heap
O(WV + E)

Each edge relaxed once → O(E)
Each bucket scanned at most once → O(WV)

Normal Dijkstra:
Find the smallest distance using a heap

Bucket version:
I already know distances are small integers, just index directly


s → a (1)
s → b (2)
a → c (2)
b → c (1)
c → t (1)

W⋅∣V∣ = 2*5 = 10

We create:
B[0], B[1], B[2], ..., B[10]
Each bucket holds vertices with that exact distance.

Step 0: B[0]: [s]

Step 1:
B[1]: [a]
B[2]: [b]

Step 2:
B[2]: [b]
B[3]: [c]

Step 3:
B[3]: [c]

Step 4:
B[4]: [t]
s:0, a:1, b:2, c:3, t:4

We place a vertex into a bucket when we discover (or improve) its shortest known distance.
Think of it like this

Buckets are indexed by distance:
B[k] = all vertices whose current best distance is k

whenever we discover a path of length k to a node, we drop it into bucket k

You don't pre-fill bins
You drop a letter into bin k when you know its priority is k
"""
def dijkstra_bucket(G, w, s, W):
    import math
    
    dist = {v: math.inf for v in G['V']}
    pred = {v: None for v in G['V']}
    
    dist[s] = 0
    
    # Buckets from 0 to W*|V|
    B = [[] for _ in range(W * len(G['V']) + 1)]
    B[0].append(s)
    
    for i in range(len(B)):
        while B[i]:
            u = B[i].pop()
            
            if dist[u] < i:
                continue  # outdated
            
            for v in G['adj'][u]:
                if dist[v] > dist[u] + w[(u,v)]:
                    old = dist[v]
                    dist[v] = dist[u] + w[(u,v)]
                    pred[v] = u
                    
                    B[dist[v]].append(v)
    
    return dist, pred

In [26]:
"""
By grouping distance estimates into exponentially growing ranges instead of exact values,
we reduce the number of buckets from O(WV) to O(log W), giving O((V+E) log W) time.

| Method                 | Structure                  | Complexity     |
| ---------------------- | -------------------------- | -------------- |
| Simple bucket Dijkstra | exact distance buckets     | O(WV + E)      |
| buket ranges w/ heaps  | logarithmic buckets + heap | O((V+E) log W) |
| standard Dijkstra      | binary heap                | O((V+E) log V) |

Level i covers: [2^i, 2^(i+1) − 1]

| Level | Range  |
| ----- | ------ |
| 0     | [0]    |
| 1     | [1]    |
| 2     | [2–3]  |
| 3     | [4–7]  |
| 4     | [8–15] |
| ...   | ...    |

Each level doubles the range:

Level 1: very precise (1 value)
Level 2: small range (2 values)
Level 3: bigger range (4 values)
Level 4: even bigger (8 values)

So we "zoom out" gradually.

If we used:
[0–3], [4–7], [8–11], ...

That would still be O(W), not O(log W).

We want:
fewer and fewer buckets as values grow

Why powers of 2 specifically?

Because they give:
clean partitioning
logarithmic number of levels
easy indexing via bit length

In fact:
bucket level ≈ floor(log2(distance))

Instead of asking:
What exact number is this?

We ask:
How many bits do I need to represent it?

That's why:
1–1 → small bucket
2–3 → next bucket
4–7 → next
8–15 → next
"""

'\nBy grouping distance estimates into exponentially growing ranges instead of exact values,\nwe reduce the number of buckets from O(WV) to O(log W), giving O((V+E) log W) time.\n\n| Method                 | Structure                  | Complexity     |\n| ---------------------- | -------------------------- | -------------- |\n| Simple bucket Dijkstra | exact distance buckets     | O(WV + E)      |\n| buket ranges w/ heaps  | logarithmic buckets + heap | O((V+E) log W) |\n| standard Dijkstra      | binary heap                | O((V+E) log V) |\n'

In [ ]:
"""
We maintain buckets indexed by:
distance mod K

Data structure:
array of buckets of size 2C (constant)
circular scanning pointer

distances are tightly bounded and grow smoothly
So we can replace the heap with a constant-size rotating structure.

If weights were arbitrary:
distances explode
buckets become huge → O(WV)

But here:
max/min weight ratio = 2
So growth is controlled.
"""
def dijkstra_fast(G, s, C):
    dist = {v: float('inf') for v in G.V}
    dist[s] = 0

    B = [[] for _ in range(2*C + 1)]
    B[0].append(s)

    i = 0  # current bucket index

    while True:
        while i < len(B) and not B[i]:
            i = (i + 1) % len(B)

        if i >= len(B):
            break

        u = B[i].pop()

        for v in G.adj[u]:
            w = weight(u,v)

            if dist[v] > dist[u] + w:
                old_bucket = dist[v] % len(B) if dist[v] != float('inf') else None
                dist[v] = dist[u] + w
                B[dist[v] % len(B)].append(v)

    return dist